# Hijje: checking the rules against spellings people wrote

Compares the project's spelling rules with the Urdu part of the **Dakshina** dataset: Roman spellings of 30,000 Urdu words, and 9,759 Urdu sentences written out in Roman letters, all by native speakers.

**Before running:** switch *Internet* on in the notebook settings. No GPU is needed.

**Dataset:** Roark et al. (2020), "Processing South Asian Languages Written in the Latin Script: the Dakshina Dataset". Licence CC BY-SA 4.0. Cite the paper if you use these results.

This notebook was written but **not run**: the machine it was written on cannot download the dataset. If a cell fails, the error message is the thing to send back.

In [ ]:
REPO_URL = "https://github.com/mirzayasirabdullahbaig07/Hijje"

import os
if not os.path.exists("pyproject.toml"):        # not inside the project folder yet
    if not os.path.isdir("hijje_repo"):
        !git clone {REPO_URL} hijje_repo
    %cd hijje_repo
!pip install -q -e .
!python -c "import hijje; print('hijje', hijje.__version__)"   # must print 0.5.0 or later

## Download the dataset and unpack the Urdu files

The download is one large file for twelve languages. Only the Urdu lexicon and the Urdu romanized sentences are unpacked.

In [ ]:
URL = "https://storage.googleapis.com/gresearch/dakshina/dakshina_dataset_v1.0.tar"
!mkdir -p ../dakshina
!wget -nv -c -P ../dakshina {URL}          # large file: this can take a few minutes with no output
!ls -lh ../dakshina/dakshina_dataset_v1.0.tar
!tar -xf ../dakshina/dakshina_dataset_v1.0.tar -C ../dakshina --wildcards "*/ur/lexicons/*" "*/ur/romanized/*"
!find ../dakshina -name "*.tsv" | sort

In [ ]:
import glob
LEX = sorted(glob.glob("../dakshina/**/ur/lexicons/*.tsv", recursive=True))
SENT = sorted(glob.glob("../dakshina/**/ur/romanized/*rejoined.aligned.cased_nopunct.tsv", recursive=True))
print(len(LEX), "lexicon files:", LEX)
print(len(SENT), "aligned sentence files:", SENT)
LEX_ARGS, SENT_ARGS = " ".join(LEX), " ".join(SENT)

# Look at the first lines, so that a format surprise is visible before any numbers are computed.
for path in LEX[:1] + SENT[:1]:
    print("\n==>", path)
    with open(path, encoding="utf-8") as fh:
        for _ in range(6):
            print(fh.readline().rstrip("\n"))

## Check 1: the word list

Each Urdu word comes with the Roman spellings that annotators gave for it. The count next to a spelling is how many times it was given, not how often people use it.

In [ ]:
WORDS = "pani theek tareeqa sawal chawal roti saath kitni hai nahi mujhe"
!python -m hijje attested --pairs {LEX_ARGS} --out results/dakshina_lexicon --show {WORDS} > /dev/null

from IPython.display import Markdown, display
display(Markdown(open("results/dakshina_lexicon/attested.md", encoding="utf-8").read()))

## Check 2: words inside sentences

The same check on words as they were written inside full sentences. Here the counts are real usage counts within these 9,759 sentences.

In [ ]:
!python -m hijje attested --pairs {SENT_ARGS} --out results/dakshina_sentences --show {WORDS} > /dev/null
display(Markdown(open("results/dakshina_sentences/attested.md", encoding="utf-8").read()))

## Keep the results

Run the cell below, then download `dakshina_results.zip` from the Output panel. Free sessions are deleted when they time out.

How to read the tables: a rule with a low share is not proven wrong, because only a few people spelled each word. Compare the rules with each other. The writers may not be from Pakistan, and they wrote carefully, so chat short forms such as "hy" or "kr" will mostly be missing.

In [ ]:
!zip -qr ../dakshina_results.zip results/dakshina_lexicon results/dakshina_sentences
!ls -lh ../dakshina_results.zip